# Notebook 04 — Modelo Preditivo: Previsao de Pico de Consumo
**Projeto:** Steel Plant Analytics
**Autor:** Ariel Marquezin
**Stack:** Scikit-learn · pandas · numpy

## Objetivo
Prever se a proxima leitura sera um pico de consumo (anomalia),
permitindo acao preventiva na gestao energetica da planta.

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn pyarrow -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
    'axes.edgecolor': '#8b949e', 'axes.labelcolor': '#ffffff',
    'axes.titlecolor': '#ffffff', 'xtick.color': '#ffffff',
    'ytick.color': '#ffffff', 'text.color': '#ffffff',
    'grid.color': '#30363d', 'grid.alpha': 0.5,
    'font.family': 'monospace', 'font.size': 12,
    'axes.titlesize': 15, 'axes.labelsize': 13,
    'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
})

C_CYAN  = '#00e5ff'; C_GREEN = '#10b981'
C_RED   = '#ef4444'; C_YELLOW = '#f59e0b'
C_BLUE  = '#3b82f6'

GOLD_PATH    = '/content/drive/MyDrive/steel/gold/'
FIGURES_PATH = '/content/drive/MyDrive/steel/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

df = pq.read_table(os.path.join(GOLD_PATH, 'steel_gold_completo.parquet')).to_pandas()
energy_col = [c for c in df.columns if 'kwh' in c.lower() or 'consumption' in c.lower()][0]
print(f'Dataset: {len(df):,} registros')
print(f'Taxa de anomalia: {df["anomalia"].mean()*100:.2f}%')

In [ ]:
# Feature Engineering
df_model = df.copy()

# Lag features (consumo anterior)
df_model['consumo_lag1']  = df_model[energy_col].shift(1)
df_model['consumo_lag4']  = df_model[energy_col].shift(4)   # 1 hora antes
df_model['consumo_lag96'] = df_model[energy_col].shift(96)  # 24 horas antes

# Rolling features
df_model['media_1h']    = df_model[energy_col].rolling(4).mean()
df_model['desvio_1h']   = df_model[energy_col].rolling(4).std()
df_model['media_24h']   = df_model[energy_col].rolling(96).mean()

# Features temporais
df_model['hora_sin']    = np.sin(2 * np.pi * df_model['hora'] / 24)
df_model['hora_cos']    = np.cos(2 * np.pi * df_model['hora'] / 24)
df_model['dia_sin']     = np.sin(2 * np.pi * df_model['dia_semana'] / 7)
df_model['dia_cos']     = np.cos(2 * np.pi * df_model['dia_semana'] / 7)

FEATURES_NUM = [
    energy_col, 'consumo_lag1', 'consumo_lag4', 'consumo_lag96',
    'media_1h', 'desvio_1h', 'media_24h',
    'hora_sin', 'hora_cos', 'dia_sin', 'dia_cos',
    'hora', 'dia_semana', 'mes', 'is_fim_semana'
]
FEATURES_CAT = ['turno']
TARGET = 'anomalia'

df_model = df_model[FEATURES_NUM + FEATURES_CAT + [TARGET]].dropna()
print(f'Dataset de modelagem: {len(df_model):,} registros')
print(f'Taxa de anomalia: {df_model[TARGET].mean()*100:.2f}%')

In [ ]:
# Split e Pre-processamento
X = df_model[FEATURES_NUM + FEATURES_CAT]
y = df_model[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), FEATURES_CAT),
])

print(f'Treino : {len(X_train):,}')
print(f'Teste  : {len(X_test):,}')

In [ ]:
# Treinar 3 modelos
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=500, class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=8, class_weight='balanced',
        random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=120, max_depth=4, learning_rate=0.08, random_state=42
    ),
}

pipelines = {}
results   = {}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, clf in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', clf)])
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]
    auc     = roc_auc_score(y_test, y_proba)
    ap      = average_precision_score(y_test, y_proba)
    cv_auc  = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
    pipelines[name] = pipe
    results[name]   = {'pipeline': pipe, 'y_pred': y_pred, 'y_proba': y_proba,
                       'roc_auc': auc, 'avg_precision': ap, 'cv_auc': cv_auc}
    print(f'{name:<25} | ROC-AUC: {auc:.4f} | CV-AUC: {cv_auc:.4f} | Avg Prec: {ap:.4f}')

best_name = max(results, key=lambda k: results[k]['roc_auc'])
best = results[best_name]
print(f'\nMelhor modelo: {best_name} (ROC-AUC = {best["roc_auc"]:.4f})')

In [ ]:
# Visualizacoes do Modelo
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0d1117')
fig.suptitle(f'Avaliacao do Modelo — {best_name}',
             fontsize=14, fontweight='bold', color='white', y=1.01)
colors_roc = [C_CYAN, C_GREEN, C_YELLOW]

ax = axes[0, 0]
ax.set_facecolor('#161b22')
for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name} (AUC={res['roc_auc']:.3f})")
ax.plot([0,1],[0,1], '--', color='#30363d', linewidth=1)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Curvas ROC', color='white')
ax.legend(fontsize=9, facecolor='#161b22', edgecolor='#30363d', labelcolor='white')
ax.grid(alpha=0.2)

ax = axes[0, 1]
ax.set_facecolor('#161b22')
for (name, res), color in zip(results.items(), colors_roc):
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    ax.plot(rec, prec, color=color, linewidth=2,
            label=f"{name} (AP={res['avg_precision']:.3f})")
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Curvas Precision-Recall', color='white')
ax.legend(fontsize=9, facecolor='#161b22', edgecolor='#30363d', labelcolor='white')
ax.grid(alpha=0.2)

ax = axes[1, 0]
ax.set_facecolor('#161b22')
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Anomalia'],
            yticklabels=['Normal','Anomalia'],
            ax=ax, cbar=False, annot_kws={'size': 14, 'weight': 'bold'})
ax.set_title('Matriz de Confusao', color='white')
ax.set_xlabel('Predito'); ax.set_ylabel('Real')

ax = axes[1, 1]
ax.set_facecolor('#161b22')
mask0 = y_test == 0; mask1 = y_test == 1
ax.hist(best['y_proba'][mask0], bins=40, alpha=0.7, color=C_GREEN, label='Normal', density=True)
ax.hist(best['y_proba'][mask1], bins=40, alpha=0.7, color=C_RED, label='Anomalia', density=True)
ax.axvline(0.5, color='white', linestyle='--', linewidth=1.5, label='Threshold 0.5')
ax.set_xlabel('Probabilidade Predita')
ax.set_ylabel('Densidade')
ax.set_title('Distribuicao de Probabilidades', color='white')
ax.legend(fontsize=9, facecolor='#161b22', edgecolor='#30363d', labelcolor='white')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '07_model_evaluation.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 7 salva')

In [ ]:
# Feature Importance
rf_pipe     = pipelines['Random Forest']
rf_clf      = rf_pipe.named_steps['classifier']
prep        = rf_pipe.named_steps['preprocessor']
cat_encoder = prep.named_transformers_['cat']
cat_feats   = cat_encoder.get_feature_names_out(FEATURES_CAT).tolist()
all_feats   = FEATURES_NUM + cat_feats
importances = pd.Series(rf_clf.feature_importances_, index=all_feats)
top15       = importances.nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(10, 7))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')
colors_fi = [C_CYAN if i >= len(top15)-5 else C_BLUE for i in range(len(top15))]
top15.plot.barh(ax=ax, color=colors_fi, edgecolor='#0d1117')
ax.set_title('Feature Importance — Random Forest\n(Top 15 preditores de anomalia)',
             fontsize=12, fontweight='bold', color='white', pad=12)
ax.set_xlabel('Importancia Relativa', fontsize=10)
ax.tick_params(labelsize=10, colors='white')
ax.grid(axis='x', alpha=0.25)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_PATH, '08_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figura 8 salva')

In [ ]:
# Simulacao de Impacto Operacional
df_test_sim = X_test.copy()
df_test_sim['y_real']  = y_test.values
df_test_sim['y_proba'] = best['y_proba']
df_test_sim['y_pred']  = best['y_pred']

verdadeiros_picos = df_test_sim[(df_test_sim['y_proba'] >= 0.5) & (df_test_sim['y_real'] == 1)]
picos_totais      = df_test_sim[df_test_sim['y_real'] == 1]
consumo_pico_total = picos_totais[energy_col].sum()
consumo_antecipado = verdadeiros_picos[energy_col].sum()
pct_capturado      = consumo_antecipado / consumo_pico_total * 100

# Simulacao de custo (R$ 0.70 por kWh em horario de pico)
custo_por_kwh     = 0.70
economia_potencial = consumo_antecipado * custo_por_kwh * 0.15  # 15% de reducao possivel

print('=' * 60)
print(' SIMULACAO DE IMPACTO OPERACIONAL')
print('=' * 60)
print(f'  Picos detectados no teste    : {len(picos_totais):,}')
print(f'  Picos identificados p/ modelo: {len(verdadeiros_picos):,}')
print(f'  % de picos capturados        : {pct_capturado:.1f}%')
print(f'  Consumo em picos capturados  : {consumo_antecipado:.1f} kWh')
print(f'  Economia potencial estimada  : R$ {economia_potencial:,.2f}')
print(f'  (assumindo R$ {custo_por_kwh}/kWh e 15% de reducao possivel com antecipacao)')
print('=' * 60)
print('\nProjeto completo! Pipeline Steel Plant Analytics finalizado.')